# JavaScript — Forms & dynamic lists

> **How this topic works**
> 1. **This notebook** — read the theory. Two lessons here: **LESSON 46**, the loop you
>    will build in the exercise, and **LESSON 47**, background on rendering a list
>    without rebuilding it.
> 2. **`project/`** — a real Vite app where you do the exercise in the browser.
>
> Read this first, then follow the steps at the bottom. DOM snippets are shown as
> plain code blocks (this kernel has no browser); runnable cells are marked.

## LESSON 46 — Forms and dynamic lists

This is the shape of almost every real app screen:

```
    DATA  →  render()  →  the page
      ↑                      │
      └──── an event ────────┘
```

1. some **data** lives in a variable
2. a **render** function draws that data into the page
3. events **change the data**, then call render again

Never patch the page by hand in ten different places. Change the data, re-render. **This is how React thinks**, minus the framework — get it here and React becomes a syntax change rather than a new way of thinking.

### Forms

```js
form.addEventListener("submit", (event) => {
  event.preventDefault();          // or the page reloads
  const value = input.value.trim();
  // ...validate, update data, re-render
  input.value = "";                // clear the field
});
```

### Rendering a list

```js
function render() {
  list.innerHTML = "";                       // clear, then rebuild

  todos.forEach((todo, index) => {
    const item = document.createElement("li");
    item.textContent = todo;
    item.dataset.index = index;              // remember which one this is
    list.append(item);
  });
}
```

### Parking your own data on an element — `data-*`

An element can carry data of your own, in an attribute whose name starts with `data-`. `item.dataset.index = index` writes `data-index="0"` into the HTML, and `event.target.dataset.index` reads it back — which is how a click finds its way to the right item in the array.

The catch is that attributes hold **strings, and nothing else**. You write the number `0` and a string `"0"` is stored, so what comes back is a string too. Wrap it in `Number()` before using it as an index:

```js
const index = Number(event.target.dataset.index);
```

### Key notes

- **Listen for `submit` on the form, not `click` on the button.** Only `submit` catches the user pressing Enter in the field.
- **Always `event.preventDefault()`.** Without it the page reloads and every variable your script held is gone.
- `input.value` is always a **string**, even with `type="number"`.
- **`dataset` stores strings both ways.** Assigning a number converts it on the way in, so reading it back always gives you a string.
- **The `tagName` check is enough only while the row is a bare `<li>`.** That is the case here, so `event.target` *is* the `<li>`. Give a row a `<span>` inside it and the click lands on the span instead — that is when you reach for `closest("li")`, as in LESSON 45.
- Rebuilding the whole list on every change feels wasteful, but is exactly right at this scale. It also fixes stale indexes for free.

### The data → render loop — runnable

The loop has nothing to do with the DOM, so you can watch it here. `render()`
returns text instead of touching a page; in `project/` it creates `<li>` elements.
Notice that **nothing ever edits the output directly** — every change goes through
the data.

In [ ]:
const todos = [];

function render() {
  if (todos.length === 0) return "(empty)";
  return todos.map((todo, index) => `${index}: ${todo}`).join("\n");
}

function addTodo(text) {
  const value = text.trim();
  if (value === "") return "Type something first.";  // validation
  todos.push(value);                                  // change the DATA
  return render();                                    // then re-render
}

function removeTodo(index) {
  todos.splice(index, 1);
  return render();
}

console.log(render());
console.log("--");
console.log(addTodo("buy bread"));
console.log("--");
console.log(addTodo("  "));        // rejected
console.log("--");
console.log(addTodo("call Mia"));
console.log("--");
console.log(removeTodo(0));        // indexes shift — re-rendering handles it

## LESSON 47 — Rendering without rebuilding

LESSON 46 rebuilds the whole list on every change. That is the right default: one code path, no chance of the page and the data disagreeing.

It has a cost, though. Emptying a `<ul>` throws away every element inside it. With bare `<li>` rows there is nothing to lose, but as soon as a row holds a checkbox, a link or a text field, the rebuild discards whatever the user had focused or selected mid-edit. The list also collapses and grows back, so the page can jump under the reader. On a long list it shows.

### insertAdjacentHTML — adding without replacing

```js
list.insertAdjacentHTML("beforeend", `<li>${text}</li>`);
```

| position | where the HTML lands |
|---|---|
| `"beforebegin"` | just before the element |
| `"afterbegin"` | inside, as the first child |
| `"beforeend"` | inside, as the last child |
| `"afterend"` | just after the element |

It parses a string into real elements and leaves the existing children untouched — that is the point of it. Nothing already on the page is thrown away and rebuilt.

It has the same catch as `innerHTML`: a string that came from a user can carry markup. Never interpolate text you did not create — use `textContent` for that.

### template — markup kept out of your JavaScript

A `<template>` holds markup the browser parses but does not display, so a row's shape stays in the HTML where it belongs.

```html
<template id="row">
  <li class="task"><span class="label"></span></li>
</template>
```

```js
const template = document.querySelector("#row");

function makeRow(text) {
  const row = template.content.cloneNode(true);   // true = copy the children too
  row.querySelector(".label").textContent = text;
  return row;
}

list.append(makeRow("Buy milk"));
```

`cloneNode(true)` is what you append — never `template.content` itself, or the second call finds an empty template.

### Which to use

Rebuild everything until it is visibly a problem. Then add rows with `insertAdjacentHTML` or a `<template>`, and keep removal by delegation. Get it correct first; make it fast only once you can point at what is slow.

One thing changes when you switch to that template, though: its row wraps the text in a `<span>`, so a click lands on the span and the `tagName === "LI"` check from LESSON 46 quietly stops matching. That is precisely the case `closest("li")` exists for.

### Key notes

- **`cloneNode(true)` copies the children; `cloneNode()` alone does not.** Forget the `true` on `template.content` and you clone an empty box: the `.label` you go looking for next is not in it, so the line after throws `Cannot set properties of null`. That error means the `true` is missing far more often than it means your selector is wrong.
- **`insertAdjacentHTML` parses HTML.** User text goes in with `textContent`, never inside the string.
- **The four names split two and two.** `afterbegin` and `beforeend` put the HTML *inside* the element; `beforebegin` and `afterend` put it *outside*, next to the element.
- Appending a template's content **moves** it. That is why you clone.

### The four positions, and clone vs move — runnable

No browser here, so this cell builds a miniature DOM out of plain objects: an element is
a name plus a list of children. Watch where each of the four positions puts the new
node — two of them go inside the list, two of them next to it — and what happens when
you insert a node you did **not** copy first.

In [ ]:
// A miniature DOM, two levels deep: a list holds rows, a row holds a name.
// Enough to see where each position lands, without a browser.
function element(name, children = []) {
  return { name, children };
}

// Print one level of children, then the tree around them.
function showChildren(node) {
  if (node.children.length === 0) return node.name;
  const inside = node.children.map((child) => child.name).join(" ");
  return `${node.name}(${inside})`;
}

// insertAdjacent, on this toy tree. The two "inside" positions touch the
// element's own children; the two "outside" ones touch its PARENT's children.
function insertAdjacent(parent, target, position, node) {
  const at = parent.children.indexOf(target);

  if (position === "afterbegin") target.children.unshift(node);
  if (position === "beforeend") target.children.push(node);
  if (position === "beforebegin") parent.children.splice(at, 0, node);
  if (position === "afterend") parent.children.splice(at + 1, 0, node);
}

// Try each position on a fresh copy of the same starting tree.
function demo(position) {
  const list = element("ul", [element("a"), element("b")]);
  const page = element("main", [list]);

  insertAdjacent(page, list, position, element("NEW"));

  // What the <ul> holds, and what sits next to it inside <main>.
  console.log(
    position + ":",
    "ul holds [" + list.children.map((child) => child.name).join(" ") + "]",
    "| main holds [" + page.children.map((child) => child.name).join(" ") + "]"
  );
}

demo("beforebegin");
demo("afterbegin");
demo("beforeend");
demo("afterend");

// --- clone vs move ------------------------------------------------------
// cloneNode(true) copies a node AND its children; without true you get the
// node alone, with nothing inside it.
function cloneNode(node, deep) {
  return element(node.name, deep ? [...node.children] : []);
}

const template = element("template", [element("li", [element("span")])]);
const row = template.children[0];

console.log("\ndeep clone    ->", showChildren(cloneNode(row, true)));
console.log("shallow clone ->", showChildren(cloneNode(row, false)), "<- the span is gone");

// Appending the template's own content MOVES it: the template is left empty,
// so the second row never arrives. This is the bug cloning prevents.
const list = element("ul");
list.children.push(...template.children.splice(0));
console.log("\nfirst append  ->", showChildren(list), "| template now holds:", template.children.length);
list.children.push(...template.children.splice(0));
console.log("second append ->", showChildren(list), "<- nothing was added the second time");

---

## Now do the exercise

The exercise covers **LESSON 46** — the data → render loop. LESSON 47 is background
reading: you will recognise `insertAdjacentHTML` and `<template>` when you meet them,
and you rebuild the list here, which is the right default at this size.

**1. Start the project**

```bash
cd project
npm install     # only the first time
npm run dev
```

**2. Read the live demo**

Open `project/src/lessons/lesson-46-forms-lists.js`. It builds the *colours* box
with exactly the loop above. Add a colour, then click one to remove it. Try adding a
colour twice: the demo refuses it. That is a second validation rule on top of the empty
check, there to show that validating is just more `if`s before the data changes — your
exercise only needs the empty one.

**3. Do the exercise**

Open `project/src/exercise/exercise.js` and work through the **6 numbered STEPs**,
each with a `// Check:` line. STEP 1 selects the elements and creates the array,
STEP 2 writes `render()`, STEPs 3-4 handle the submit and reject an empty task,
STEP 5 removes by delegation, and STEP 6 renders once on load.
You're rebuilding the same machine for a to-do list.

**Done when:** you can add tasks, an empty submit is refused with a message, and
clicking a task removes it — including after several adds and removes.

Stuck? Paste `ai-prompt.txt` into a fresh AI session. The answer is in
`project/src/exercise/solution.js` — last resort.